# Anthropic API

> Translate Anthropic Messages API requests, responses, and streams to and from fastllm's shared message types, with tool replay, media, and prompt caching

In [ ]:
#| default_exp anthropic

## Imports

In [ ]:
#| export
import json, copy
from collections import Counter
from fastcore.utils import *
from fastcore.meta import *
from fasttransport.errors import api_error_from_event

from fastllm.types import *
from aidialog.msg_parts import (MdStr, Part, PartType, mk_part, Msg, Completion, Text, Thinking, ToolUse, ToolResult, tool_text,
    InputImage, InputAudio, InputVideo, InputFile, data_url)
from fastllm.streaming import *
from fastllm.streaming import mk_acollect_stream

In [ ]:
#| hide
from fasttransport.core import AsyncHttpCli
from fastcore.test import *
from fastcore.funccall import get_schema
from cachy.core import enable_cachy, disable_cachy, doms
from aidialog.msg_parts import fmt2hist, hist2fmt
from fastllm.acomplete import acomplete, defaults


In [ ]:
enable_cachy(doms=doms+sample_doms, hdrs=('content-type',))


In [ ]:
#| hide
defaults.reasoning_effort = 'l'

`ant_call` posts a request body to `/v1/messages` through `provider_req`, just as `acomplete` does. With `stream=True`, it returns an event iterator instead of a complete response.

In [ ]:
ant_cli = AsyncHttpCli('https://api.anthropic.com', base_headers={"x-api-key": os.environ["ANTHROPIC_API_KEY"], "anthropic-version": "2023-06-01"})
async def ant_call(**body): return await provider_req(ant_cli, '/v1/messages', body, stream=body.get('stream', False))

The harness pins the `anthropic-version` header to `2023-06-01`. This is an API version, not a model name.

## Normalize

The normalizers build shared `ToolUse`, `Usage`, and message parts from Anthropic responses. `mk_completion` combines them into a `Completion`.

### Tool Calls

A `tool_use` block asks the client to run a tool. `server_tool_use` and `mcp_tool_use` describe server-side calls, so their `ToolUse` parts have `server=True`. All three carry an ID, name, and input dict. Extra fields go in `raw`.

In [ ]:
def lite_mk_func(f):
    if isinstance(f, dict): return f
    return {'type':'function', 'function':get_schema(f, pname='parameters')}

In [ ]:
def simple_add(a:int,b:int):
    'add numbers'
    return a + b

In [ ]:
sch = lite_mk_func(simple_add)
sch

{'type': 'function',
 'function': {'name': 'simple_add',
  'description': 'add numbers',
  'parameters': {'type': 'object',
   'properties': {'a': {'description': '', 'type': 'integer'},
    'b': {'description': '', 'type': 'integer'}},
   'required': ['a', 'b']}}}

In [ ]:
#| export
ant_tc_types = ("tool_use", "server_tool_use", "mcp_tool_use")

def norm_tool_call(b):
    if b.get("type") in ant_tc_types:
        raw = {k:v for k,v in b.items() if k not in ("type","id","name","input")}
        return ToolUse(id=b["id"], name=b["name"], arguments=b.get("input") or {}, server=b.get("type")!="tool_use", raw=raw)

def norm_tool_calls(resp, delta=False):
    "Extract Anthropic tool_use blocks as normalized tool calls."
    out = []
    for b in resp.get("content", []):
        if tc:= norm_tool_call(b): out.append(tc)
    return out

User defined:

In [ ]:
mn,mt = sonn,200

In [ ]:
resp = await ant_call(model=mn, max_tokens=mt, messages=[{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}], tools=[dict(name="simple_add", description="add numbers", input_schema=sch['function']['parameters'])])
norm_tool_calls(resp)

[ToolUse(raw={'caller': {'type': 'direct'}}, cache_control=None, id='toolu_01XtrGeRqzjUmMBCQUqd1qoY', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, text=None),
 ToolUse(raw={'caller': {'type': 'direct'}}, cache_control=None, id='toolu_017DmvweGDJCc69sE27PhKtt', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, text=None)]

Web search:

In [ ]:
resp = await ant_call(model=mn, max_tokens=mt, messages=[{"role": "user", "content": "Use web search to get the weather in Istanbul"}], tools=[{"type": "web_search_20250305", "name": "web_search"}])
norm_tool_calls(resp)

[ToolUse(raw={}, cache_control=None, id='srvtoolu_01Vn86yansnaZdqSVDqhj1gz', name='web_search', arguments={'query': 'Istanbul weather today'}, server=True, text=None)]

MCP:

In [ ]:
norm_tool_calls({"content": [dict(type="mcp_tool_use", id="mcptoolu_abc123", name="get_weather", input={"city": "Istanbul"}, server_name="weather-server")]})

[ToolUse(raw={'server_name': 'weather-server'}, cache_control=None, id='mcptoolu_abc123', name='get_weather', arguments={'city': 'Istanbul'}, server=True, text=None)]

### Usage

Anthropic reports uncached input, cache reads, and cache creation separately. `norm_usage` adds them to get `prompt_tokens` and retains the separate counts. It also records hosted web-search requests.

After collecting the message, `finalize_usage` splits output tokens into completion and reasoning tokens. It uses `output_tokens_details.thinking_tokens` when nonzero. Otherwise it estimates 1.5 tokens per word of thinking text, capped at total output tokens. The total stays unchanged.

In [ ]:
#| export
def norm_usage(resp):
    "Normalize Anthropic usage shape."
    if not (usg:=resp.get("usage")): return None
    cached = int(usg.get("cache_read_input_tokens", 0) or 0)
    cache_creation = int(usg.get("cache_creation_input_tokens", 0) or 0)
    pt = int(usg.get("input_tokens", 0) or 0) + cached + cache_creation
    ct = int(usg.get("output_tokens", 0) or 0)
    return Usage(prompt_tokens=pt, completion_tokens=ct, total_tokens=pt + ct,
        cached_tokens=cached, cache_creation_tokens=cache_creation, reasoning_tokens=0, raw=usg,
        search_queries=int(nested_idx(usg, 'server_tool_use', 'web_search_requests') or 0))

def finalize_usage(usg, parts):
    "Split reasoning tokens out of `output_tokens`: the reported `thinking_tokens`, else an estimate from the thinking text."
    if not usg: return usg
    rc = '\n'.join(p.text or '' for p in parts if isinstance(p, Thinking))
    ct = int(usg.raw.get('output_tokens', usg.completion_tokens) or 0)
    rt = int(nested_idx(usg.raw, 'output_tokens_details', 'thinking_tokens') or 0) or (min(int(len(rc.split())*1.5), ct) if rc else 0)
    return Usage(prompt_tokens=usg.prompt_tokens, completion_tokens=ct-rt, total_tokens=usg.prompt_tokens+ct,
        cached_tokens=usg.cached_tokens, cache_creation_tokens=usg.cache_creation_tokens, reasoning_tokens=rt, search_queries=usg.search_queries, raw=usg.raw)

Text response:

In [ ]:
resp = await ant_call(model=mn, max_tokens=mt, messages=[{"role": "user", "content": "Hi"}])
norm_usage(resp)

Usage(prompt_tokens=11, completion_tokens=14, total_tokens=25, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'input_tokens': 11, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 14, 'output_tokens_details': {'thinking_tokens': 0}, 'service_tier': 'standard', 'inference_geo': 'global'})

Tool call response:

In [ ]:
resp = await ant_call(model=mn, max_tokens=mt, messages=[{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}], tools=[dict(name="simple_add", description="add numbers", input_schema=sch['function']['parameters'])])
norm_usage(resp)

Usage(prompt_tokens=418, completion_tokens=135, total_tokens=553, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'input_tokens': 418, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 135, 'output_tokens_details': {'thinking_tokens': 0}, 'service_tier': 'standard', 'inference_geo': 'global'})

Server tools:

In [ ]:
resp = await ant_call(model=mn, max_tokens=mt, messages=[{"role": "user", "content": "Use web search to get the weather in Istanbul"}], tools=[{"type": "web_search_20250305", "name": "web_search"}])
norm_usage(resp)

Usage(prompt_tokens=11408, completion_tokens=254, total_tokens=11662, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=1, raw={'input_tokens': 11408, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 254, 'output_tokens_details': {'thinking_tokens': 200}, 'service_tier': 'standard', 'inference_geo': 'global', 'server_tool_use': {'web_search_requests': 1, 'web_fetch_requests': 0}})

### Finish Reason

`end_turn`, `tool_use`, and `max_tokens` become `stop`, `tool_calls`, and `length`. Other reasons pass through. A stopped response with a client tool call becomes `tool_calls`; server calls don't need a client reply.

In [ ]:
#| export
def norm_finish(resp, tcs=None):
    "Canonicalize finish_reason to OpenAI Chat values: stop, tool_calls, length, content_filter."
    reason = resp.get("stop_reason")
    mp = dict(end_turn=FinishReason.stop, tool_use=FinishReason.tool_calls, max_tokens=FinishReason.length)
    r =  mp.get(reason, reason)
    return FinishReason.tool_calls if r==FinishReason.stop and any(~L(tcs).attrgot('server')) else r 

In [ ]:
norm_finish(resp)

<finish_reason.length: 'length'>

### Non-stream Completion

Hosted search returns citations in text blocks and sources in a `web_search_tool_result` block. `norm_parts` converts citations with URLs to `url_citation` entries. `search_results` matches each result to its server call by ID. It puts the sources on that call's `text` as Markdown `title: url` lines, or records the error code for a failed search.

This gives the call a result that we can display and store. The original result block, including encrypted page content, stays in the raw message for replay.


In [ ]:
#| export
def search_results(parts, blocks):
    "Give each server call the sources its `web_search_tool_result` block reports, one `title: url` line each, or the block's error code"
    calls = {p.id: p for p in parts if isinstance(p, ToolUse) and p.server}
    for b in blocks:
        if b.get('type') != 'web_search_tool_result' or not (call := calls.get(b.get('tool_use_id'))): continue
        res = b.get('content')
        call.text = res.get('error_code', '') if isinstance(res, dict) else MdStr('\n'.join(f"{r.get('title', '')}: {r['url']}" for r in res if r.get('url')))
    return parts

In [ ]:
#| export
def _ant_part_type(typ):
    "Map Anthropic content block type to canonical PartType."
    ant_parts_mapping = dict(text=PartType.text, thinking=PartType.thinking, redacted_thinking=PartType.thinking, tool_use=PartType.tool_use, server_tool_use=PartType.tool_use, mcp_tool_use=PartType.tool_use)
    return ant_parts_mapping.get(typ, typ)

def _url_cite(c): return dict(type='url_citation', url=c['url'], title=c.get('title', ''))

def norm_parts(resp):
    "Canonical parts of a response; a text block's web citations become `url_citation`s"
    parts = []
    for b in resp.get("content", []):
        typ = _ant_part_type(b.get("type", "text"))
        if   typ == PartType.thinking: parts.append(Thinking(b.get("thinking", ""), raw=b))
        elif typ == PartType.tool_use:
            if tc := norm_tool_call(b): parts.append(tc)
        elif typ == PartType.text:
            cites = [_url_cite(c) for c in b.get('citations') or [] if c.get('url')]
            parts.append(Text(b.get("text", ""), citations=cites, raw=b))
    return search_results(parts, resp.get("content", []))

`raw_msg` wraps the response's content list in an assistant message. `Msg.raw` retains the original blocks, including thinking signatures and server-tool results. Those blocks matter when continuing a tool loop.

In [ ]:
#| export
def raw_msg(resp):
    "The assistant message as the API sent it"
    return dict(role='assistant', content=resp['content'])

Tool results live in user messages, outside the assistant response that `norm_parts` reads. `norm_tr_parts` converts their content to shared parts. A string or text block becomes `Text`. An image becomes `InputImage`: a URL source keeps its URL, and a base64 source supplies its `media_type` and `data`. Neither conversion fetches the image. An unsupported image source type raises `ValueError`. Other block types go through the `Part` registry. Registered extensions can convert them; unknown types raise an error.

In [ ]:
#| export
def norm_tr_parts(content):
    "Normalize an Anthropic `tool_result` content str or block list to canonical `Part`s"
    if isinstance(content, str): return [Text(content)]
    parts = []
    for b in content:
        typ = b.get('type', 'text')
        if typ=='text': parts.append(Text(b.get('text',''), raw=b))
        elif typ=='image':
            s = b['source']
            if s['type']=='url': parts.append(InputImage(s['url'], raw=b))
            elif s['type']=='base64': parts.append(InputImage(f"data:{s['media_type']};base64,{s['data']}", mime=s['media_type'], raw=b))
            else: raise ValueError(f"Unsupported Anthropic image source type: {s['type']!r}")
        else: parts.append(mk_part(**b))
    return parts

In [ ]:
tp = norm_tr_parts([dict(type='text', text='The login page.'),
    dict(type='image', source=dict(type='base64', media_type='image/png', data='aGk=')),
    dict(type='image', source=dict(type='url', url='https://example.com/login.png'))])
test_eq(type(tp[0]), Text)
test_eq(tp[1].mime, 'image/png')
test_eq(tp[2].text, 'https://example.com/login.png')
test_eq(norm_tr_parts('plain')[0].text, 'plain')
with expect_fail(KeyError): norm_tr_parts([dict(type='mystery')])
tp

[Text(raw={'type': 'text', 'text': 'The login page.'}, cache_control=None, text='The login page.', citations=None),
 InputImage(raw={'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/png', 'data': 'aGk='}}, cache_control=None, text='data:image/png;base64,aGk=', mime='image/png'),
 InputImage(raw={'type': 'image', 'source': {'type': 'url', 'url': 'https://example.com/login.png'}}, cache_control=None, text='https://example.com/login.png', mime=None)]

#### Text

In [ ]:
api_name,vnd_nm = 'anthropic', 'anthropic'

In [ ]:
# Not exported - later we add an extra param
api_registry.register('anthropic', norm_tool_calls=norm_tool_calls, norm_parts=norm_parts, norm_finish=norm_finish, norm_usage=norm_usage, finalize_usage=finalize_usage, raw_msg=raw_msg)

In [ ]:
resp = await ant_call(model=mn, max_tokens=mt, messages=[{"role": "user", "content": "Say hi"}])
comp = mk_completion(resp, mn, api_name, vnd_nm)
comp

Hi there! 👋 How can I help you today?

::: details

- model: `claude-sonnet-5-5`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=12, completion_tokens=18, total_tokens=30, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'input_tokens': 12, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 18, 'output_tokens_details': {'thinking_tokens': 0}, 'service_tier': 'standard', 'inference_geo': 'global'})`

:::

#### Thinking

In [ ]:
comp = mk_completion({"model": "claude-sonnet-4-20250514", "content": [
    dict(type="thinking", thinking="Let me reason...", text=""),
    {"type": "redacted_thinking", "text": ""},
    {"type": "text", "text": "Here's my answer."}
], "stop_reason": "end_turn", "usage": {"input_tokens": 10, "output_tokens": 20}}, mn, api_name, vnd_nm)
comp


::: details
## Thinking

Let me reason...

:::

Here's my answer.

::: details

- model: `claude-sonnet-5-5`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=10, completion_tokens=16, total_tokens=30, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=4, search_queries=0, raw={'input_tokens': 10, 'output_tokens': 20})`

:::

#### Tool call

In [ ]:
resp = await ant_call(model=mn, max_tokens=mt, messages=[{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}], tools=[dict(name="simple_add", description="add numbers", input_schema=sch['function']['parameters'])])
comp = mk_completion(resp, mn, api_name, vnd_nm)
comp



🔧 simple_add({'a': 3, 'b': 5})


🔧 simple_add({'a': 10, 'b': 20})


::: details

- model: `claude-sonnet-5-5`
- finish_reason: `tool_calls`
- usage: `Usage(prompt_tokens=418, completion_tokens=135, total_tokens=553, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'input_tokens': 418, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 135, 'output_tokens_details': {'thinking_tokens': 0}, 'service_tier': 'standard', 'inference_geo': 'global'})`

:::

In [ ]:
test_eq(comp.tool_calls[0].server, False)

Web search response:

In [ ]:
resp = await ant_call(model=mn, max_tokens=mt, messages=[{"role": "user", "content": "Use web search to get the weather in Istanbul"}], tools=[{"type": "web_search_20250305", "name": "web_search"}])
comp = mk_completion(resp, mn, api_name, vnd_nm)
assert comp.usage.search_queries >= 1, comp.usage
comp



🔧 web_search({'query': 'Istanbul weather today'})


::: details

- model: `claude-sonnet-5-5`
- finish_reason: `length`
- usage: `Usage(prompt_tokens=11408, completion_tokens=54, total_tokens=11662, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=200, search_queries=1, raw={'input_tokens': 11408, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 254, 'output_tokens_details': {'thinking_tokens': 200}, 'service_tier': 'standard', 'inference_geo': 'global', 'server_tool_use': {'web_search_requests': 1, 'web_fetch_requests': 0}})`

:::

In [ ]:
test_eq(comp.tool_calls[0].server, True)
assert 'http' in comp.tool_calls[0].text
comp.tool_calls[0].text

'Istanbul, Istanbul, Türkiye Weather Forecast | AccuWeather: https://www.accuweather.com/en/tr/istanbul/318251/weather-forecast/318251\nIstanbul, TR Weather Forecast, Conditions, and Maps – Yahoo Weather: https://weather.yahoo.com/tr/istanbul/istanbul/\nIstanbul, Turkey 14 day weather forecast: https://www.timeanddate.com/weather/turkey/istanbul/ext\n10-Day Weather Forecast for Istanbul, Istanbul, Türkiye - The Weather Channel | weather.com: https://weather.com/tr/istanbul/city/istanbul/tenday\nClimate of Istanbul: https://en.wikipedia.org/wiki/Climate_of_Istanbul\nIstanbul, Türkiye Weather Forecast | MSN Weather: https://www.msn.com/en-xl/weather/forecast/in-Istanbul,%20T%C3%BCrkiye\nIstanbul, Türkei Weather Forecast | MSN Weather: https://www.msn.com/en-xl/weather/forecast/in-Istanbul,T%C3%BCrkei\nNew on Yahoo: https://weather.yahoo.com/tr/bartin/ulus\nIstanbul (Turkey) weather: https://weather.metoffice.gov.uk/forecast/sxk972u3h'

MCP (mocked):

In [ ]:
comp = mk_completion(dict(model="claude-sonnet-4-20250514",
    content=[dict(type="mcp_tool_use", id="mcptoolu_abc", name="get_weather", input={"city": "Istanbul"}, server_name="weather-server")],
    stop_reason="tool_use", usage={"input_tokens": 50, "output_tokens": 30}), mn, api_name, vnd_nm)
comp



🔧 get_weather({'city': 'Istanbul'})


::: details

- model: `claude-sonnet-5-5`
- finish_reason: `tool_calls`
- usage: `Usage(prompt_tokens=50, completion_tokens=30, total_tokens=80, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'input_tokens': 50, 'output_tokens': 30})`

:::

In [ ]:
test_eq(comp.tool_calls[0].server, True)

### Streaming

`norm_sse_event` turns Anthropic events into shared `Delta` objects. `PartAccum` collects the parts while `acollect_stream` yields text, thinking, and citation updates. The final result is a `Completion`, as in the non-streaming path.

In [ ]:
print_stream = PrintStream()

Anthropic starts a tool call with `content_block_start` and sends its arguments as `input_json_delta` fragments. We put each fragment in `ToolUse.arguments` under `_delta` for the collector to join:

```py
ToolUse(id=str(ev.get("index", "")), name="", arguments={"_delta": d.get("partial_json", '')})
```

This differs from Responses, whose `response.output_item.done` event supplies a complete call. The other branches below read text, thinking, and citation deltas. `message_delta` supplies finish reason and usage; `error` raises an API error.

In [ ]:
#| export
def norm_sse_event(ev, **kwargs):
    ev = obj2dict(ev)
    typ = ev.get("type")
    text, thinking, tcs, citations = None, None, [], None
    if typ == "content_block_start":
        cb = ev.get("content_block", {})
        if tc := norm_tool_call(cb):
            if not tc.arguments: tc.arguments = {'_delta': ''}
            tcs = [tc]
    elif typ == "content_block_delta":
        d = ev.get("delta", {})
        dtyp = d.get("type")
        if   dtyp == "text_delta": text = d.get("text")
        elif dtyp == "thinking_delta": thinking = d.get("thinking")
        elif dtyp == "input_json_delta":
            tcs = [ToolUse(id=str(ev.get("index", "")), name="", arguments={"_delta": d.get("partial_json", '')})]
        elif dtyp == "citations_delta": citations = [_url_cite(c) for c in listify(d.get('citation', [])) if c.get('url')]
    elif typ == "message_delta":
        fin = norm_finish(ev.get('delta'))
        return Delta(finish_reason=fin, usage=norm_usage(ev), raw=ev, **kwargs)
    elif typ == "error": raise api_error_from_event(ev)
    return Delta(text=text, thinking=thinking, tool_calls=tcs, citations=citations, raw=ev, **kwargs)

`PartAccum` asks `delta_index_fn` which part to update. The function returns the current part index and any index state needed for the next event.

The shared callback accepts the current `Delta`, its part type, and the previous type and index. Anthropic supplies a block index in `d.raw`, so we don't need that previous state.

Events for one content block share an index. This keeps each tool's argument fragments together.

In [ ]:
#| export
def delta_index_fn(d, typ, last_typ, last_idx):
    'Returns accumulation index for current delta and updated last idx'
    return nested_idx(d, 'raw', 'index'), None

`collate_raw` rebuilds an assistant message for replay. It copies each starting block, appends text and thinking fragments, records signatures, and collects citations. It parses a tool's accumulated JSON when the block stops. The resulting content blocks serve the same purpose as `raw_msg` on an ordinary response.

In [ ]:
#| export
def collate_raw(deltas):
    "The assistant message rebuilt from a stream's events"
    blocks = {}
    for ev in L(deltas).attrgot('raw'):
        typ,i = ev.get('type'),ev.get('index')
        if typ == 'content_block_start': blocks[i] = copy.deepcopy(ev['content_block'])
        elif typ == 'content_block_delta':
            d,b = ev['delta'],blocks[i]
            if   d['type'] == 'text_delta':       b['text'] += d['text']
            elif d['type'] == 'thinking_delta':   b['thinking'] += d['thinking']
            elif d['type'] == 'signature_delta':  b['signature'] = d['signature']
            elif d['type'] == 'input_json_delta': b['_json'] = b.get('_json', '') + d['partial_json']
            elif d['type'] == 'citations_delta':  b['citations'] = (b.get('citations') or []) + [d['citation']]
        elif typ == 'content_block_stop' and '_json' in blocks[i]: blocks[i]['input'] = json.loads(blocks[i].pop('_json') or '{}')
    return dict(role='assistant', content=list(blocks.values()))

The Anthropic collector also attaches search results to server calls before yielding the final `Completion`.

In [ ]:
#| export
@delegates(mk_acollect_stream, but=['index_fn', 'api_name'])
async def acollect_stream(resp, **kwargs):
    res = mk_acollect_stream(norm_and_yield(resp, norm_sse_event), index_fn=delta_index_fn, api_name='anthropic', **kwargs)
    async for o in res:
        if isinstance(o, Completion): search_results(o.message.content, o.message.raw['content'])
        yield o

In [ ]:
api_registry.register('anthropic', norm_tool_calls=norm_tool_calls, norm_parts=norm_parts, norm_finish=norm_finish, norm_usage=norm_usage, finalize_usage=finalize_usage, acollect_stream=acollect_stream, raw_msg=raw_msg, collate_raw=collate_raw)

Let's compare streamed and non-streamed thinking, tool calls, and web search.

#### Thinking

In [ ]:
msgs, eff = [{"role": "user", "content": "What is 31231231 * 12312?"}], {"type": "adaptive", "display": "summarized"}
resp = await ant_call(model=mn, messages=msgs, max_tokens=8000, thinking=eff)
comp = mk_completion(resp, mn, api_name, vnd_nm)
comp


::: details
## Thinking

I'm breaking 31231231*12312 into parts: multiplying by 12000 and by 312 separately, then adding them to get 384,518,916,072, and double-checking the smaller multiplication of 31231231*12 to verify.



:::

31231231 × 12312 = **384,518,916,072**

Split 12312 into 12000 + 312:

- 31,231,231 × 12,000 = 374,774,772,000
- 31,231,231 × 312 = 9,744,144,072
- Sum: 374,774,772,000 + 9,744,144,072 = 384,518,916,072

::: details

- model: `claude-sonnet-5-5`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=20, completion_tokens=117, total_tokens=323, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=186, search_queries=0, raw={'input_tokens': 20, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 303, 'output_tokens_details': {'thinking_tokens': 186}, 'service_tier': 'standard', 'inference_geo': 'global'})`

:::

In [ ]:
test_eq(comp.usage.reasoning_tokens > 0, True)
test_eq(comp.usage.completion_tokens + comp.usage.reasoning_tokens, comp.usage.raw['output_tokens'])


In [ ]:
resp = await ant_call(model=mn,messages=msgs,max_tokens=8000,thinking=eff,stream=True)
async for o in acollect_stream(resp): print_stream(o)

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠31231231 × 12312 = **384,518,916,072**

Split 12312 = 12000 + 312:
- 31231231 × 12000 = 374,774,772,000
- 31231231 × 312 = 9,744,144,072
- Sum: 374,774,772,000 + 9,744,144,072 = 384,518,916,072

In [ ]:
test_eq(o.usage.reasoning_tokens > 0, True)
test_eq(o.usage.completion_tokens + o.usage.reasoning_tokens, o.usage.raw['output_tokens'])


In [ ]:
o


::: details
## Thinking

I'm splitting 31231231*12312 into 12000 and 312 parts, computing each product separately, then adding them to get 384,518,916,072, and double-checking the 31231231*12 subcomponent.



:::

31231231 × 12312 = **384,518,916,072**

Split 12312 = 12000 + 312:
- 31231231 × 12000 = 374,774,772,000
- 31231231 × 312 = 9,744,144,072
- Sum: 374,774,772,000 + 9,744,144,072 = 384,518,916,072

::: details

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=20, completion_tokens=112, total_tokens=324, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=192, search_queries=0, raw={'input_tokens': 20, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 304, 'output_tokens_details': {'thinking_tokens': 192}})`

:::

In [ ]:
test_eq('thinking' in L(comp.message.content).attrgot('type'), True)
test_eq('thinking' in L(o.message.content).attrgot('type'), True)

In [ ]:
test_eq(len(comp.message.content), len(o.message.content))
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.tool_calls, o.tool_calls)

#### Tool call

In [ ]:
msgs = [{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}]
tools = [dict(name="simple_add", description="add numbers", input_schema=sch['function']['parameters'])]
resp = await ant_call(model=mn, messages=msgs, tools=tools, max_tokens=8000, thinking=eff)
comp = mk_completion(resp, mn, api_name, vnd_nm)
comp



🔧 simple_add({'a': 3, 'b': 5})


🔧 simple_add({'a': 10, 'b': 20})


::: details

- model: `claude-sonnet-5-5`
- finish_reason: `tool_calls`
- usage: `Usage(prompt_tokens=418, completion_tokens=135, total_tokens=553, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'input_tokens': 418, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 135, 'output_tokens_details': {'thinking_tokens': 0}, 'service_tier': 'standard', 'inference_geo': 'global'})`

:::

In [ ]:
resp = await ant_call(model=mn, messages=msgs, tools=tools, max_tokens=8000, thinking=eff, stream=True)
async for o in acollect_stream(resp): print_stream(o)


- ⏳ `simple_add(a="3", b="5")` ⏳

- ⏳ `simple_add(a="10", b="20")` ⏳


In [ ]:
def test_tcs(tcs1,tcs2):
    for tc1,tc2 in zip(tcs1,tcs2): 
        test_eq(tc1.name,tc2.name)
        test_eq(tc1.arguments,tc2.arguments)
        test_eq(tc1.server,tc2.server)

In [ ]:
test_tcs(comp.tool_calls,o.tool_calls)

In [ ]:
test_eq(len(comp.message.content), len(o.message.content))
test_eq(comp.finish_reason, o.finish_reason)
test_eq(o.message.raw['content'][-1]['input'], o.tool_calls[-1].arguments)
test_eq([b['type'] for b in o.message.raw['content']], [p.type for p in o.message.content])
comp.usage, o.usage

(Usage(prompt_tokens=418, completion_tokens=135, total_tokens=553, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'input_tokens': 418, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 135, 'output_tokens_details': {'thinking_tokens': 0}, 'service_tier': 'standard', 'inference_geo': 'global'}),
 Usage(prompt_tokens=418, completion_tokens=135, total_tokens=553, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'input_tokens': 418, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 135, 'output_tokens_details': {'thinking_tokens': 0}}))

#### Web search

In [ ]:
def add_ant_link(o):
    "Render a citations-bearing delta `Part` as markdown links (the chat layer's `postproc` does this in fastllm)"
    if not (isinstance(o, Text) and o.citations): return o
    links = []
    for c in o.citations:
        if 'title' not in c: return o
        title = c['title'].replace('"', '\\"')
        links.append(f'[*]({c["url"]} "{title}")')
    return Text(' '.join(links))

In [ ]:
msgs = [{"role": "user", "content": "Use web search to get the weather in Istanbul?"}]
tools=[{"type": "web_search_20250305", "name": "web_search"}]
resp = await ant_call(model=mn, messages=msgs, tools=tools, max_tokens=8000, thinking=eff)
comp = mk_completion(resp, mn, api_name, vnd_nm)
comp



🔧 web_search({'query': 'Istanbul weather today'})
Istanbul is cloudy, windy and rainy today (Monday, Sept 28).

- **Right now:** It was 64°F (feels like 57°F) at 4:39 AM, with light rain and a NNE wind at 16 mph, gusting to 26 mph. Another source, using the Atatürk Airport station, showed 18 °C with rain showers.
- **Today's forecast:** Showers, with a high of 68°F and a low of 62°F, and a 90% chance of precipitation. AccuWeather expects morning showers, then cloudy and windy conditions. It also expects rain and drizzle in the afternoon.
- **Tonight:** Breezy at times this evening, otherwise cloudy with a few showers late.
- **Air quality:** Fair.

Bring an umbrella and a light jacket. In Celsius, the high is about 20°C and the low about 17°C.

::: details

- model: `claude-sonnet-5-5`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=11410, completion_tokens=489, total_tokens=11899, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=1, raw={'input_tokens': 11410, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 489, 'output_tokens_details': {'thinking_tokens': 0}, 'service_tier': 'standard', 'inference_geo': 'global', 'server_tool_use': {'web_search_requests': 1, 'web_fetch_requests': 0}})`

:::

In [ ]:
resp = await ant_call(model=mn, messages=msgs, tools=tools, max_tokens=8000, thinking=eff, stream=True)
async for o in acollect_stream(resp): print_stream(add_ant_link(o))

Istanbul is cloudy, windy and showery today, Monday, September 28.

- **Now:** [*](https://www.accuweather.com/en/tr/istanbul/318251/weather-forecast/318251 "Istanbul, Istanbul, Türkiye Weather Forecast | AccuWeather")AccuWeather shows 64°F with light rain, a RealFeel of 57°, and NNE wind at 16 mph with gusts up to 26 mph. Timeanddate, using the Istanbul Atatürk Airport station, [*](https://www.timeanddate.com/weather/turkey/istanbul/ext "Istanbul, Turkey 14 day weather forecast")reports 18 °C with rain showers.
- **Today's forecast:** [*](https://www.accuweather.com/en/tr/istanbul/318251/weather-forecast/318251 "Istanbul, Istanbul, Türkiye Weather Forecast | AccuWeather")Morning showers, otherwise cloudy and windy, with a high of 68°F. [*](https://weather.yahoo.com/tr/istanbul/istanbul/ "Istanbul, TR Weather Forecast, Conditions, and Maps – Yahoo Weather")Yahoo gives a low of 62°F and a 90% chance of precipitation.
- **Tonight:** [*](https://www.accuweather.com/en/tr/istanbul/318251/w

In [ ]:
o



🔧 web_search({'query': 'Istanbul weather today'})
Istanbul is cloudy, windy and showery today, Monday, September 28.

- **Now:** AccuWeather shows 64°F with light rain, a RealFeel of 57°, and NNE wind at 16 mph with gusts up to 26 mph. Timeanddate, using the Istanbul Atatürk Airport station, reports 18 °C with rain showers.
- **Today's forecast:** Morning showers, otherwise cloudy and windy, with a high of 68°F. Yahoo gives a low of 62°F and a 90% chance of precipitation.
- **Tonight:** It will be breezy at times this evening, then cloudy with a few showers late. AccuWeather also lists rain and drizzle in the afternoon.
- **Air quality:** AccuWeather rates it fair.

In Celsius, that's roughly 18°C now, a high near 20°C and a low near 17°C. Take an umbrella and a light jacket.

::: details

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=11410, completion_tokens=509, total_tokens=11919, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=1, raw={'input_tokens': 11410, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 509, 'output_tokens_details': {'thinking_tokens': 0}, 'server_tool_use': {'web_search_requests': 1, 'web_fetch_requests': 0}})`

:::

In [ ]:
test_eq([(t.name, t.server) for t in comp.tool_calls], [(t.name, t.server) for t in o.tool_calls])
assert 'http' in o.tool_calls[0].text
test_eq(comp.finish_reason, o.finish_reason)
comp.usage, o.usage

(Usage(prompt_tokens=11410, completion_tokens=489, total_tokens=11899, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=1, raw={'input_tokens': 11410, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 489, 'output_tokens_details': {'thinking_tokens': 0}, 'service_tier': 'standard', 'inference_geo': 'global', 'server_tool_use': {'web_search_requests': 1, 'web_fetch_requests': 0}}),
 Usage(prompt_tokens=11410, completion_tokens=509, total_tokens=11919, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=1, raw={'input_tokens': 11410, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 509, 'output_tokens_details': {'thinking_tokens': 0}, 'server_tool_use': {'web_search_requests': 1, 'web_fetch_requests': 0}}))

## Denormalize

The request converters turn shared messages and options into Anthropic fields.

### Msgs

In [ ]:
def mk_user_msg(txt): return Msg(role='user', content=[Text(txt)])

User messages contain input blocks. Assistant messages contain replies and tool calls. A tool reply becomes a user message containing `tool_result` blocks.

In [ ]:
#| export
def _ant_cc(block, p):
    "Add the part's prompt-cache directive to `block`, if it has one."
    if p.cache_control: block['cache_control'] = p.cache_control
    return block

`_sanid` replaces characters outside `[a-zA-Z0-9_-]` with underscores in tool IDs. Both tool calls and results use it so their IDs still match.

In [ ]:
#| export
def _sanid(id_str): return re.sub(r'[^a-zA-Z0-9_-]', '_', id_str or '')

This first tool-result converter handles text. The exported version below adds media.

In [ ]:
def denorm_tool_result(p:ToolResult):
    "Convert canonical tool_result Part to Anthropic tool_result content block."
    return _ant_cc(dict(type='tool_result', tool_use_id=_sanid(p.id), content=tool_text(p.text)), p)

This first user-message converter handles text. The exported version below adds media inputs.

In [ ]:
def denorm_user(m:Msg):
    "Convert canonical user Msg to Anthropic user message."
    parts = [_ant_cc(dict(type='text', text=p.text or ''), p) for p in m.content if isinstance(p, Text)]
    return dict(role='user', content=parts)

In [ ]:
#| export
def denorm_tool_use(p:ToolUse):
    "Convert canonical tool_use Part to Anthropic tool_use content block."
    return _ant_cc(dict(type='tool_use', id=_sanid(p.id or ''), name=p.name or '', input=p.arguments or {}), p)

def denorm_assistant(m:Msg):
    "The message as the API sent it, else its text and tool calls; a cache mark on the last part is copied onto the last raw block"
    if m.raw:
        cc = [p.cache_control for p in m.content if p.cache_control]
        if not cc: return m.raw
        return m.raw | dict(content=m.raw['content'][:-1] + [m.raw['content'][-1] | dict(cache_control=cc[-1])])
    blocks = [_ant_cc(dict(type='text', text=p.text or ''), p) if isinstance(p, Text) else denorm_tool_use(p) for p in m.content if isinstance(p, (Text, ToolUse))]
    return dict(role='assistant', content=blocks)

def denorm_tool(m:Msg):
    "Convert canonical tool Msg to Anthropic user message with tool_result blocks."
    blocks = [denorm_tool_result(p) for p in m.content if isinstance(p, ToolResult)]
    return dict(role='user', content=blocks)

def denorm_msgs(msgs:list[Msg]):
    "Convert list of canonical Msgs to Anthropic messages."
    res = []
    for m in msgs:
        if   m.role == 'user':      res.append(denorm_user(m))
        elif m.role == 'assistant': res.append(denorm_assistant(m))
        elif m.role == 'tool':      res.append(denorm_tool(m))
    return res

When an assistant message has `raw`, `denorm_assistant` reuses it. Signed thinking, search calls, and encrypted results survive the round trip. There is one exception: a cache mark on a canonical part can add `cache_control` to the last raw block. That creates a new message dict rather than modifying `raw`.

In [ ]:
turn = [mk_user_msg('Use web search to get the weather in Istanbul?'), comp.message]
wire = denorm_msgs(turn)
test_is(wire[1], comp.message.raw)
[b['type'] for b in wire[1]['content']]

['server_tool_use',
 'web_search_tool_result',
 'text',
 'text',
 'text',
 'text',
 'text',
 'text',
 'text',
 'text',
 'text',
 'text',
 'text',
 'text',
 'text',
 'text',
 'text']

The `hist2fmt`/`fmt2hist` round trip drops `raw`. We then reconstruct text and tool calls without signed thinking or encrypted results. A stored search becomes an ordinary call/result pair. This example sends that history back for a later turn.

In [ ]:
stored = fmt2hist(hist2fmt([comp.message]))
wire = denorm_msgs([turn[0], *stored, mk_user_msg('And in Ankara? One sentence.')])
test_eq([b['type'] for b in wire[1]['content']], ['text', 'tool_use'])
test_eq(wire[2]['content'][0]['type'], 'tool_result')
resp = await ant_call(model=mn, messages=wire, tools=tools, max_tokens=8000, thinking=eff)
mk_completion(resp, mn, api_name, vnd_nm).message.text

"Before Ankara, I need to correct my Istanbul answer. That search returned only page titles and links, with no weather data. The temperatures, wind speeds and rain chances I gave weren't from the results, and I shouldn't have presented them as fact. Please disregard them. I can rerun the Istanbul search if you'd like.\n\nAnkara today: cloudy with occasional light rain and a high near 60°F (about 16°C), with a low around 55°F tonight."

### Tool Schemas

In [ ]:
#| export
def denorm_tool_schs(tools):
    "Convert canonical tools to Anthropic format."
    out = []
    for t in tools:
        fn = fn_schema(t)
        if fn is None: out.append(t); continue
        name, desc, params = fn
        out.append(dict(name=name, description=desc, input_schema=params))
    return out

In [ ]:
test_eq(denorm_tool_schs([sch]), [dict(name='simple_add', description='add numbers', input_schema=sch['function']['parameters'])])
test_eq(denorm_tool_schs([{"type": "web_search_20250305", "name": "web_search"}]), [{"type": "web_search_20250305", "name": "web_search"}])

### Tool Choice

In [ ]:
#| export
def denorm_tool_choice(v):
    "Map canonical tool_choice to Anthropic format."
    if v is None: return None
    if v in ('auto',):                    return {'type': 'auto'}
    if v in ('required', 'any', 'force'): return {'type': 'any'}
    if v in ('none', 'off', 'disabled'):  return {'type': 'none'}
    return {'type': 'tool', 'name': v}

Modes

In [ ]:
test_eq(denorm_tool_choice('auto'), {'type': 'auto'})
test_eq(denorm_tool_choice('required'), {'type': 'any'})
test_eq(denorm_tool_choice('none'), {'type': 'none'})

Tool name

In [ ]:
test_eq(denorm_tool_choice('simple_add'), {'type': 'tool', 'name': 'simple_add'})

None

In [ ]:
test_eq(denorm_tool_choice(None), None)

### Reasoning Effort

String efforts select adaptive thinking. `minimal` maps to `low`; the other accepted values are `low`, `medium`, `high`, `xhigh`, and `max`. Use a model that supports the requested adaptive-thinking configuration. Older models may require a different configuration.

There is no `disable` string option here. `None` adds no configuration, while a dict passes through unchanged. A caller can use that dict form for explicit provider settings, including `{'thinking': {'type': 'disabled'}}` when the model supports it.

In [ ]:
#| export
def denorm_reasoning(v):
    "Map canonical reasoning_effort to Anthropic adaptive thinking + output_config."
    mp = dict(minimal='low', low='low', medium='medium', high='high', xhigh='xhigh', max='max')
    err = ValueError(f"Invalid reasoning effort for Anthropic: {v}, accepted string values are: {list(mp)} and dicts are passthrough")
    if v is None: return None
    elif isinstance(v, dict): return v
    elif isinstance(v, str) and v in mp:  return {"thinking": {"type": "adaptive"}, "output_config": {"effort": mp.get(v)}}
    raise err

In [ ]:
test_eq(denorm_reasoning('low'), {"thinking": {"type": "adaptive"}, "output_config": {"effort": "low"}})
test_eq(denorm_reasoning('high'), {"thinking": {"type": "adaptive"}, "output_config": {"effort": "high"}})
test_eq(denorm_reasoning(None), None)

In [ ]:
denorm_reasoning(dict(thinking={"type":"adaptive", "display":"summarized"}, output_config={"effort":'high'}))

{'thinking': {'type': 'adaptive', 'display': 'summarized'},
 'output_config': {'effort': 'high'}}

### Web Search Options

In [ ]:
#| export
def denorm_web_search(v):
    "Map canonical web_search_options to Anthropic hosted web_search tool."
    _max_uses = dict(low=1, medium=5, high=10)
    t = {"type": "web_search_20250305", "name": "web_search"}
    if (typ := (v or {}).get("type")): t["type"] = typ
    if (s := (v or {}).get("search_context_size")): t["max_uses"] = _max_uses.get(s, 5)
    if (u := (v or {}).get("user_location", {}).get("approximate")): t["user_location"] = {"type": "approximate", **u}
    return t

None / empty

In [ ]:
test_eq(denorm_web_search(None), {"type": "web_search_20250305", "name": "web_search"})

search_context_size

In [ ]:
test_eq(denorm_web_search({"search_context_size": "medium"})["max_uses"], 5)

user_location

In [ ]:
loc = {"approximate": {"city": "Istanbul"}}
test_eq(denorm_web_search({"user_location": loc})["user_location"], {"type": "approximate", "city": "Istanbul"})

### System

In [ ]:
#| export
def denorm_system(sp):
    if isinstance(sp, Msg): sp = sp.content[0]
    if isinstance(sp, Part): return [_ant_cc(dict(type='text', text=sp.text), sp)]
    else: return sp

In [ ]:
sp = 'You are a pirate. Always respond in pirate speak. Keep it to one sentence.'
msg1 = mk_user_msg('What are you?')

`denorm_system` puts the prompt in `system`, not in the message list. A `Part` can carry a cache mark; a `Msg` supplies its first part. The example marks a repeated system prompt and checks the reported cache tokens.

In [ ]:
ant_sp = Text(sp*200, cache_control={"type": "ephemeral"})

In [ ]:
resp = await ant_call(model=mn,system=denorm_system(ant_sp), messages=denorm_msgs([msg1]),max_tokens=8000,stream=True)
async for comp in acollect_stream(resp): pass
comp

Arr, I be a swashbucklin' AI matey, sailin' the digital seas to help ye with whatever ye need, savvy?

::: details

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=5613, completion_tokens=46, total_tokens=5659, cached_tokens=0, cache_creation_tokens=5603, reasoning_tokens=0, search_queries=0, raw={'input_tokens': 10, 'cache_creation_input_tokens': 5603, 'cache_read_input_tokens': 0, 'output_tokens': 46, 'output_tokens_details': {'thinking_tokens': 0}})`

:::

In [ ]:
test_eq(comp.usage.cache_creation_tokens>0, True)

### Media Inputs

The shared media classes store URLs or data URLs in `text` and the media type in `mime`. Anthropic's user-message converter accepts `InputImage` and `InputFile`. It rejects `InputAudio` and `InputVideo`.

The media classes describe the input kind, not a promise that every provider or model accepts it. Common formats include JPEG, PNG, GIF, and WebP images; WAV and MP3 audio; MP4 and WebM video; and PDF, DOCX, or TXT files. Check the selected provider and model before sending a format.

The current user-message converters use these representations:

| Part | Responses | Chat | Anthropic | Gemini |
|---|---|---|---|---|
| Image URL or data URL | `input_image.image_url` | `image_url.url` | `image.source` | `inlineData` or `fileData` |
| Base64 audio | Raises `ValueError` | `input_audio` with data and format | Raises `ValueError` | `inlineData` |
| Video URL | `input_video.video_url` | Raises `ValueError` | Raises `ValueError` | `fileData` |
| File URL | `input_file.file_url` | Raises `ValueError` | `document.source` | `fileData` |
| File data URL | `input_file.file_data` and filename | `file.file_data` and filename | `document.source` | `inlineData` |

Responses images use `{"type": "input_image", "image_url": ...}`. Files use `file_url` for URLs or `file_data` plus `filename` for data URLs. The local converter also emits `input_video` for video. It rejects audio.

Chat nests an image URL in `{"type": "image_url", "image_url": {"url": ...}}`. Audio needs a base64 data URL. Its converter emits `{"type": "input_audio", "input_audio": {"data": ..., "format": ...}}` and maps WAV/MP3 MIME types to format names. Files also need base64 data URLs here. The converter doesn't accept file IDs, despite its error message mentioning them.

Anthropic uses `{"type": "image", "source": ...}` or `{"type": "document", "source": ...}`. A source contains either a URL or base64 data with `media_type`. The document examples use PDF with `application/pdf`. The converter forwards the supplied MIME type rather than enforcing a PDF-only rule. It applies `cache_control` to supported content blocks.

Gemini uses `{"inlineData": {"mimeType": ..., "data": ...}}` for data URLs. It uses `{"fileData": {"mimeType": ..., "fileUri": ...}}` for URLs, including YouTube video URLs. The API's `videoMetadata` can describe start/end offsets; this converter does not set it.

The original API comparison cited Responses [input content](https://developers.openai.com/api/reference/resources/responses/methods/create) (`specs/openai.with-code-samples.yml:68361,68429`), Chat [content parts](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create) (`35185,35217,35253`), Anthropic [image/document blocks](https://docs.anthropic.com/en/api/messages) (`specs/anthropic.yml:12159,6740`), and Gemini [Part/Blob/FileData](https://ai.google.dev/api/generate-content) (`specs/gemini.json:189-387`).


In [ ]:
#| export
def denorm_user(m:Msg):
    "Convert canonical user Msg to Anthropic user message."
    parts = []
    for p in m.content:
        if   isinstance(p, Text):       parts.append(_ant_cc({"type":"text", "text":p.text or ''}, p))
        elif isinstance(p, InputImage): parts.append(_ant_cc(denorm_image(p), p))
        elif isinstance(p, InputAudio): raise ValueError("Anthropic does not support audio input")
        elif isinstance(p, InputVideo): raise ValueError("Anthropic does not support video input")
        elif isinstance(p, InputFile):  parts.append(_ant_cc(denorm_file(p), p))
    return dict(role='user', content=parts)

#### Image

In [ ]:
#| export
def denorm_image(p, max_sz=None):
    if (b64:=data_url(p.text)):
        data = resize_b64(b64[1], max_sz) if max_sz else b64[1]
        return {"type": "image", "source": dict(type="base64", media_type=b64[0], data=data)}
    return {"type": "image", "source": {"type": "url", "url": p.text}}

Image URL

In [ ]:
msg1 = Msg(role='user', content=[InputImage(sample_img_url), Text('What is this image?')])

In [ ]:
resp = await ant_call(model=mn,messages=denorm_msgs([msg1]),max_tokens=8000,stream=True)
async for comp in acollect_stream(resp): pass
comp

This is a landscape photograph of a calm mountain lake, seen from a wooden deck or dock.

- **Foreground:** Weathered wooden planks run across the bottom of the frame. They look like a dock, pier, or deck, and they give the scene a "viewing platform" feel.
- **Middle ground:** The lake is very still, so it mirrors the trees and mountains almost perfectly. Along the far shore is a line of evergreen and deciduous trees, with a strip of golden marsh grass at the water's edge.
- **Background:** Large snow-capped mountains rise behind the trees. A soft blue haze and low clouds give the peaks a misty, dreamy look.

The scene closely resembles **Lake Matheson in New Zealand's South Island**, which is famous for its mirror-like reflections of Aoraki/Mount Cook and Mount Tasman. I can't say for sure that it's that location. The wooden planks also look a bit like a composite or stock-photo edit, since the deck may have been added to a separate landscape shot.

Images like this are often used as **desktop wallpapers, presentation backgrounds, or product-display backdrops**, because the empty deck in the foreground is a good place to put a product or text.

Are you trying to identify the location, or do you want to use the image for something specific?

::: details

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=610, completion_tokens=429, total_tokens=1039, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'input_tokens': 610, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 429, 'output_tokens_details': {'thinking_tokens': 0}})`

:::

Image data

In [ ]:
img_b64 = "data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAQAAAAECAIAAAAmkwkpAAAAEElEQVR4nGP4z8AARwzEcQCukw/x0F8jngAAAABJRU5ErkJggg=="
msg1 = Msg(role='user', content=[InputImage(img_b64), Text('What color is this pixel?')])

In [ ]:
resp = await ant_call(model=mn,messages=denorm_msgs([msg1]),max_tokens=8000,stream=True)
async for comp in acollect_stream(resp): pass
comp

The image is almost entirely **white**, with a small **red** square near the top-left. If you mean that red spot, the pixel is red, roughly pure red (about RGB 255, 0, 0). I can't measure exact values from the image, so that's an estimate.

If you meant a different pixel, tell me which one. For exact values, you could open the image in an editor like Paint, Photoshop, or GIMP and use the eyedropper tool.

::: details

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=20, completion_tokens=141, total_tokens=285, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=124, search_queries=0, raw={'input_tokens': 20, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 265, 'output_tokens_details': {'thinking_tokens': 124}})`

:::

#### Files

For base64 documents, `denorm_file` takes `media_type` from the data URL. A remote document uses a URL source instead. Unlike the OpenAI file converters, this helper doesn't add a filename. Gemini calls the corresponding type field `mimeType`.

In [ ]:
#| export
def denorm_file(p):
    if (b64:=data_url(p.text)): return {"type": "document", "source": dict(type="base64", media_type=b64[0], data=b64[1])}
    return {"type": "document", "source": {"type": "url", "url": p.text}}

In [ ]:
pdf_url = "https://arxiv.org/pdf/1706.03762"
msg1 = Msg(role='user', content=[InputFile(pdf_url), Text('What is the title of this paper?')])

In [ ]:
resp = await ant_call(model=mn,messages=denorm_msgs([msg1]),max_tokens=8000,stream=True)
async for comp in acollect_stream(resp): pass
comp

The title of this paper is **"Attention Is All You Need."** It was written by Vaswani et al. at Google Brain and Google Research, and was presented at NIPS 2017. It introduces the Transformer, a sequence transduction architecture based entirely on attention mechanisms rather than recurrence or convolutions.

::: details

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=38371, completion_tokens=104, total_tokens=38475, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'input_tokens': 38371, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 104, 'output_tokens_details': {'thinking_tokens': 0}})`

:::

### Media Tool Call Results

A list-valued tool result can contain `Text`, `InputImage`, and `InputFile` parts. Other part types raise `ValueError`. Base64 tool-result images pass through `resize_b64` with `max_sz=2000`; image URLs remain URLs. A non-list result uses `tool_text`. The result's cache mark goes on its outer `tool_result` block.

In [ ]:
#| export
def denorm_tool_result(p:ToolResult):
    "Convert canonical tool_result Part to Anthropic tool_result content block."
    tid = _sanid(p.id or (p.raw or {}).get('call_id',''))
    if isinstance(p.text, list):
        blocks = []
        for pp in p.text:
            if   isinstance(pp, Text):       blocks.append({"type": "text", "text": pp.text or ""})
            elif isinstance(pp, InputImage): blocks.append(denorm_image(pp, max_sz=2000))
            elif isinstance(pp, InputFile):  blocks.append(denorm_file(pp))
            else: raise ValueError(f"Anthropic tool_result does not support {pp.type}")
        return _ant_cc(dict(type='tool_result', tool_use_id=tid, content=blocks), p)
    return _ant_cc(dict(type='tool_result', tool_use_id=tid, content=tool_text(p.text)), p)

In [ ]:
def test_img(): "Show the test image"
img_tools = denorm_tool_schs([lite_mk_func(test_img)])
img_q = mk_user_msg('Call test_img, then describe what it shows in one sentence.')
resp = await ant_call(model=mn, messages=denorm_msgs([img_q]), tools=img_tools, max_tokens=8000)
comp = mk_completion(resp, mn, api_name, vnd_nm)
msgs = [img_q, comp.message, Msg(role='tool', content=[ToolResult(id=comp.tool_calls[0].id, name='test_img', text=[InputImage(img_b64)])])]
comp.tool_calls


[ToolUse(raw={'caller': {'type': 'direct'}}, cache_control=None, id='toolu_01BC2DcJhzzi1RE7N7WTjjRd', name='test_img', arguments={}, server=False, text=None)]

In [ ]:
resp = await ant_call(model=mn, messages=denorm_msgs(msgs), tools=img_tools, max_tokens=8000, stream=True)
async for comp in acollect_stream(resp): pass
comp


The image is a very small, mostly white square with a red patch in its top-left corner.

::: details

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=406, completion_tokens=27, total_tokens=464, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=31, search_queries=0, raw={'input_tokens': 406, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 58, 'output_tokens_details': {'thinking_tokens': 31}})`

:::

## Payload

### Prompt Caching

`cache_control` marks a prompt-cache breakpoint. `_ant_cc` copies a part's mark onto its request block, including text, media, tool calls, and tool results. The API allows [at most four breakpoints per request](https://platform.claude.com/docs/en/build-with-claude/prompt-caching).

`_add_cache_control` chooses the last `Text` or `ToolUse` part in a message. It adds an ephemeral mark, with the default five-minute lifetime or an explicit `ttl='1h'`.

In [ ]:
def _add_cache_control(
    msg,     # Canonical Msg to mark
    ttl=None # Cache TTL: '5m' (default) or '1h'
):
    "Set a cache breakpoint on the last `Text` or `ToolUse` part of `msg`"
    cc = {"type": "ephemeral"} | ({"ttl": ttl} if ttl else {})
    idxs = [i for i,p in enumerate(msg.content) if isinstance(p, (Text, ToolUse))]
    if idxs: msg.content[idxs[-1]].cache_control = cc
    return msg

In [ ]:
m = Msg('assistant', [Text('Let me check.'), ToolUse(id='t1', name='lookup')])
_add_cache_control(m)
test_eq(m.content[-1].cache_control, {'type': 'ephemeral'})
test_eq(m.content[0].cache_control, None)
m

**Msg**

- role: `assistant`

<contents>

**Text** (`text`)

Let me check.

::: details

- raw: `None`
- citations: `None`

:::

🔧 **lookup**(`{}`)

::: details

- id: `t1`
- server: `False`
- raw: `None`

:::

</contents>

`apply_cache_idxs` replaces the conversation's cache breakpoints on each call. It clears the old marks, then marks the messages at `cache_idxs`. Replacing the selection keeps stale breakpoints from accumulating past the API's limit of four.

Tool-role messages don't count toward the indices. With a system prompt, index 0 selects that prompt and positive message indices start at 1. Negative indices count back through non-tool messages. Out-of-range indices do nothing. The function returns the system prompt, wrapped and marked if selected.

In [ ]:
def _clear_cache_control(obj):
    "Clear canonical cache marks from a message or part."
    if isinstance(obj, Msg):
        for p in obj.content: _clear_cache_control(p)
    elif isinstance(obj, Part): obj.cache_control = None

def apply_cache_idxs(
    msgs,            # Canonical Msgs to mark
    cache_idxs=[-1], # Breakpoint idxs into `msgs` (tool-role msgs excluded); 0 is `sp` when given
    sp=None,         # System prompt: str, Msg, or Part
    ttl=None         # Cache TTL: '5m' (default) or '1h'
):
    "Replace cache breakpoints in `msgs` and `sp`, then return the system prompt."
    for m in msgs: _clear_cache_control(m)
    _clear_cache_control(sp)
    if sp:
        if 0 in cache_idxs:
            if isinstance(sp, str): sp = Text(sp)
            if isinstance(sp, Part): sp = Msg('', [sp])
            _add_cache_control(sp, ttl)
        cache_idxs = [o-1 if o>0 else o for o in cache_idxs if o]
    ms = [j for j,m in enumerate(msgs) if m.role != 'tool']
    for i in cache_idxs:
        try: _add_cache_control(msgs[ms[i]], ttl)
        except IndexError: continue
    return sp

In [ ]:
cmsgs = [Msg('user', [Text('question')]), Msg('assistant', [Text('answer')]),
    Msg('tool', [ToolResult(id='t1', name='lookup', text='out')]), Msg('user', [Text('follow-up')])]
csp = apply_cache_idxs(cmsgs, [0,-1], sp='You are helpful')
test_eq(csp.content[0].cache_control, {'type': 'ephemeral'})
test_eq(cmsgs[-1].content[-1].cache_control, {'type': 'ephemeral'})
test_eq(cmsgs[0].content[0].cache_control, None)
csp

**Msg**

- role: ``

<contents>

**Text** (`text`)

You are helpful

::: details

- raw: `None`
- citations: `None`

:::

</contents>

Changing the indices replaces the previous message marks. Here `-2` selects the assistant message: the tool reply between it and the final user message doesn't count.

In [ ]:
apply_cache_idxs(cmsgs, [-2])
test_eq(cmsgs[-1].content[-1].cache_control, None)
test_eq(cmsgs[1].content[-1].cache_control, {'type': 'ephemeral'})
cmsgs[1]

**Msg**

- role: `assistant`

<contents>

**Text** (`text`)

answer

::: details

- raw: `None`
- citations: `None`

:::

</contents>

`mk_payload` combines messages, model, and supported options. It adds web search to the tools list and applies `cache_idxs` before converting messages. An omitted or `None` `max_tokens` uses the model's catalogued output limit (1024 when unknown). An explicit limit takes precedence.

In [ ]:
#| export
@delegates(payload_kwargs)
def mk_payload(msgs, model, **kwargs):
    sp = kwargs.get('system')
    if (ci:=kwargs.get('cache_idxs')) is not None: sp = apply_cache_idxs(msgs, ci, sp, kwargs.get('ttl'))
    max_tokens = kwargs.get('max_tokens')
    if max_tokens is None: max_tokens = get_model_info(model, 'anthropic').get('max_output_tokens') or 1024
    payload = dict(model=model, messages=denorm_msgs(msgs), max_tokens=max_tokens)
    if kwargs.get('stream'):                payload['stream'] = True
    if sp:                              payload['system'] = denorm_system(sp)
    if tools:=kwargs.get('tools'):          payload['tools'] = denorm_tool_schs(tools)
    if tchc:=kwargs.get('tool_choice'):     payload['tool_choice'] = denorm_tool_choice(tchc)
    if thk:=kwargs.get('reasoning_effort'): payload.update(denorm_reasoning(thk))
    if (wopts:=kwargs.get('web_search_options')) is not None: payload.setdefault('tools', []).append(denorm_web_search(wopts))
    if (temp:=kwargs.get('temperature')) is not None: payload['temperature'] = temp
    return payload

In [ ]:
mk_payload([mk_user_msg('Hello')], sonn)['max_tokens']


128000

`csp` still has the system breakpoint we set earlier. Selecting only the last message removes that system mark and gives the message a one-hour breakpoint. Passing `cache_idxs=[]` then clears the canonical marks. Use `None` to leave manual marks alone.

Other provider payload builders that don't read `cache_idxs` ignore it, so callers can pass the same option across backends.

In [ ]:
p = mk_payload(cmsgs, mn, system=csp, cache_idxs=[-1], ttl='1h')
assert 'cache_control' not in p['system'][0]
test_eq(p['messages'][-1]['content'][-1]['cache_control'], {'type': 'ephemeral', 'ttl': '1h'})

p = mk_payload(cmsgs, mn, system=csp, cache_idxs=[])
assert 'cache_control' not in p['system'][0]
assert 'cache_control' not in p['messages'][-1]['content'][-1]
p['system'], p['messages'][-1]

([{'type': 'text', 'text': 'You are helpful'}],
 {'role': 'user', 'content': [{'type': 'text', 'text': 'follow-up'}]})

`get_hdrs` always sets the API version. It uses `x-api-key` for an explicit key or `ANTHROPIC_API_KEY`. An `oauth_token` instead selects bearer authentication and the `oauth-2025-04-20` beta header.

In [ ]:
#| export
def get_hdrs(api_key=None, oauth_token=None):
    "An OAuth token travels as a bearer under Anthropic's OAuth beta; an API key as `x-api-key`"
    hdrs = {"anthropic-version": "2023-06-01"}
    if oauth_token: return hdrs | {"Authorization": f"Bearer {oauth_token}", "anthropic-beta": "oauth-2025-04-20"}
    return hdrs | {"x-api-key": get_api_key(api_key, 'ANTHROPIC_API_KEY')}


The next test checks OAuth header construction with a fake token. It makes no request. A token from `claude setup-token` was the intended input for this branch; the test doesn't establish whether a subscription token currently grants access to this endpoint.

In [ ]:
oauth_hdrs = get_hdrs(oauth_token='sk-ant-oat01-x')
test_eq((oauth_hdrs['Authorization'], oauth_hdrs['anthropic-beta']), ('Bearer sk-ant-oat01-x', 'oauth-2025-04-20'))
assert 'x-api-key' not in oauth_hdrs
oauth_hdrs

{'anthropic-version': '2023-06-01',
 'Authorization': 'Bearer sk-ant-oat01-x',
 'anthropic-beta': 'oauth-2025-04-20'}

## Cost

`cost` uses raw token counts and model metadata from `get_model_info`. It charges uncached input, output, cache reads, and cache creation separately. Five-minute and one-hour cache writes use different rates. If the detailed creation counts are absent, it treats the aggregate creation count as five-minute tokens.

In [ ]:
#| export
def cost(usage, m):
    raw = usage.raw
    in_tok = raw['input_tokens']
    cache_read = raw.get('cache_read_input_tokens', 0)
    cc = raw.get('cache_creation', {}) or {}
    cache_5m  = cc.get('ephemeral_5m_input_tokens', raw.get('cache_creation_input_tokens', 0))
    cache_1h  = cc.get('ephemeral_1h_input_tokens', 0)
    cost  = in_tok     * m.input_cost_per_token
    cost += raw['output_tokens'] * m.output_cost_per_token
    cost += cache_read * m.get('cache_read_input_token_cost', 0)
    cost += cache_5m   * m.get('cache_creation_input_token_cost', 0)
    cost += cache_1h   * m.get('cache_creation_input_token_cost_above_1hr', 0)
    return cost

In [ ]:
m = dict2obj(dict(input_cost_per_token=1, output_cost_per_token=10, cache_read_input_token_cost=0.1,
    cache_creation_input_token_cost=2, cache_creation_input_token_cost_above_1hr=4))

u_flat = SimpleNamespace(raw=dict(input_tokens=10, output_tokens=5, cache_read_input_tokens=100, cache_creation_input_tokens=3))
test_close(cost(u_flat, m), 10 + 50 + 10 + 6)

u_nested = SimpleNamespace(raw=dict(input_tokens=10, output_tokens=5, cache_read_input_tokens=100,
    cache_creation=dict(ephemeral_5m_input_tokens=3, ephemeral_1h_input_tokens=2)))
test_close(cost(u_nested, m), 10 + 50 + 10 + 6 + 8)

## Register API

The final registry entry supplies response normalizers, `finalize_usage`, the payload builder, stream collector, headers, and cost function. `raw_msg` and `collate_raw` retain replay data. The endpoint is `/v1/messages`; `acomplete` appends it to the provider's base URL.

In [ ]:
#| export
api_registry.register('anthropic', norm_tool_calls=norm_tool_calls, norm_parts=norm_parts, norm_finish=norm_finish, norm_usage=norm_usage,
    finalize_usage=finalize_usage, raw_msg=raw_msg, collate_raw=collate_raw, acollect_stream=acollect_stream, mk_payload=mk_payload,
    cost=cost, get_hdrs=get_hdrs, endpoint='/v1/messages')

## Costs through `acomplete`

`acomplete` sends `claude` models through this adapter. These checks compare `Completion.cost` between streamed and non-streamed `acomplete` calls.

Text

In [ ]:
msg = mk_user_msg("Say hello in French")

In [ ]:
comp = await acomplete([msg], model=sonn, max_tokens=64)
comp.cost

0.0006680000000000001

In [ ]:
# Streaming - as a sanity check
it = await acomplete([msg], model=sonn, max_tokens=64, stream=True)
async for scomp in it: pass
test_eq(scomp.usage.prompt_tokens, comp.usage.prompt_tokens)
assert scomp.cost > 0
scomp.cost

0.0006680000000000001

Image

In [ ]:
msg = Msg('user', content=[InputImage(sample_img_url), Text('What is this image?')])
comp = await acomplete([msg], model=sonn, max_tokens=64)
comp.cost

0.00186

In [ ]:
# Streaming - as a sanity check
it = await acomplete([msg], model=sonn, max_tokens=64, stream=True)
async for scomp in it: pass
test_eq(scomp.usage.prompt_tokens, comp.usage.prompt_tokens)
assert scomp.cost > 0
scomp.cost

0.00186

Cache read and write

In [ ]:
big_text = 'The quick brown fox jumps over the lazy dog. ' * 200
msg1 = Msg('user', content=[Text(big_text, cache_control={"type": "ephemeral"}), Text('Summarize')])
comp1 = await acomplete([msg1], model=sonn, max_tokens=64)  # writes cache
msg3 = Msg('user', content=[Text('Now in French')])
comp2 = await acomplete([msg1, comp1.message, msg3], model=sonn, max_tokens=64)  # reads cache

The second request should read the cached prompt. The assertion checks that it did, then displays the cost, including the cache read.

In [ ]:
assert comp2.usage.raw['cache_read_input_tokens'] > 0  # the second call actually read the cache
comp2.cost

0.0015208

In [ ]:
# Streaming - as a sanity check
it1 = await acomplete([msg1], model=sonn, max_tokens=64, stream=True)  # writes cache
it2 = await acomplete([msg1, comp1.message, msg3], model=sonn, max_tokens=64, stream=True)  # reads cache
async for comp1 in it1: pass
async for comp2 in it2: pass

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev
nbdev.nbdev_export()